In [0]:
-- ============================================================
-- Project: Databricks SQL Retail Analytics Dashboard
-- File: 03_validation_queries.sql
-- Purpose: Reconciliation, integrity, uniqueness and quality
--          validations for the Silver and Gold layers
-- ============================================================

USE SCHEMA retail_analytics_project;


-- ============================================================
-- 1. Silver row count by transaction type
-- ============================================================

SELECT
    transaction_type,
    COUNT(*) AS total_rows,

    ROUND(
        SUM(signed_line_amount),
        2
    ) AS signed_amount,

    ROUND(
        SUM(gross_sales_amount),
        2
    ) AS gross_sales_amount,

    ROUND(
        SUM(cancellation_amount),
        2
    ) AS cancellation_amount

FROM silver_retail_sales_clean

GROUP BY
    transaction_type

ORDER BY
    total_rows DESC;


-- ============================================================
-- 2. Silver to Gold row reconciliation
-- Gold fact_sales must contain SALE and CANCELLATION only
-- ============================================================

WITH silver_commercial AS (
    SELECT
        COUNT(*) AS silver_commercial_rows
    FROM silver_retail_sales_clean
    WHERE transaction_type IN (
        'SALE',
        'CANCELLATION'
    )
),

gold_fact AS (
    SELECT
        COUNT(*) AS gold_fact_rows
    FROM fact_sales
)

SELECT
    s.silver_commercial_rows,
    g.gold_fact_rows,

    s.silver_commercial_rows
        - g.gold_fact_rows AS row_count_difference,

    CASE
        WHEN s.silver_commercial_rows = g.gold_fact_rows
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM silver_commercial s

CROSS JOIN gold_fact g;


-- ============================================================
-- 3. Reconcile excluded Silver transaction types
-- ============================================================

SELECT
    COUNT(*) AS total_silver_rows,

    SUM(
        CASE
            WHEN transaction_type IN ('SALE', 'CANCELLATION')
            THEN 1
            ELSE 0
        END
    ) AS commercial_rows,

    SUM(
        CASE
            WHEN transaction_type = 'RETURN_OR_ADJUSTMENT'
            THEN 1
            ELSE 0
        END
    ) AS inventory_adjustment_rows,

    SUM(
        CASE
            WHEN transaction_type = 'INVALID_OR_NON_COMMERCIAL'
            THEN 1
            ELSE 0
        END
    ) AS non_commercial_rows,

    COUNT(*)
    -
    SUM(
        CASE
            WHEN transaction_type = 'RETURN_OR_ADJUSTMENT'
            THEN 1
            ELSE 0
        END
    )
    -
    SUM(
        CASE
            WHEN transaction_type = 'INVALID_OR_NON_COMMERCIAL'
            THEN 1
            ELSE 0
        END
    ) AS expected_fact_rows

FROM silver_retail_sales_clean;


-- ============================================================
-- 4. Monetary reconciliation between Silver and fact_sales
-- ============================================================

WITH silver_metrics AS (
    SELECT
        ROUND(
            SUM(gross_sales_amount),
            2
        ) AS silver_gross_sales,

        ROUND(
            SUM(cancellation_amount),
            2
        ) AS silver_cancellation_amount

    FROM silver_retail_sales_clean

    WHERE transaction_type IN (
        'SALE',
        'CANCELLATION'
    )
),

gold_metrics AS (
    SELECT
        ROUND(
            SUM(gross_sales_amount),
            2
        ) AS gold_gross_sales,

        ROUND(
            SUM(cancellation_amount),
            2
        ) AS gold_cancellation_amount,

        ROUND(
            SUM(net_sales_amount),
            2
        ) AS gold_net_sales

    FROM fact_sales
)

SELECT
    s.silver_gross_sales,
    g.gold_gross_sales,

    ROUND(
        s.silver_gross_sales - g.gold_gross_sales,
        2
    ) AS gross_sales_difference,

    s.silver_cancellation_amount,
    g.gold_cancellation_amount,

    ROUND(
        s.silver_cancellation_amount
        - g.gold_cancellation_amount,
        2
    ) AS cancellation_difference,

    g.gold_net_sales,

    CASE
        WHEN s.silver_gross_sales = g.gold_gross_sales
         AND s.silver_cancellation_amount =
             g.gold_cancellation_amount
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM silver_metrics s

CROSS JOIN gold_metrics g;


-- ============================================================
-- 5. Validate net sales formula
-- ============================================================

SELECT
    COUNT(*) AS invalid_net_sales_rows

FROM fact_sales

WHERE ROUND(
    net_sales_amount,
    2
) <> ROUND(
    gross_sales_amount - cancellation_amount,
    2
);


-- ============================================================
-- 6. Validate net quantity formula
-- ============================================================

SELECT
    COUNT(*) AS invalid_net_quantity_rows

FROM fact_sales

WHERE net_quantity
    <> sold_quantity - cancelled_quantity;


-- ============================================================
-- 7. Check duplicated sales_line_id
-- Expected: zero returned rows
-- ============================================================

SELECT
    sales_line_id,
    COUNT(*) AS duplicate_count

FROM fact_sales

GROUP BY
    sales_line_id

HAVING
    COUNT(*) > 1

ORDER BY
    duplicate_count DESC;


-- ============================================================
-- 8. Check null or blank fact keys
-- ============================================================

SELECT
    SUM(
        CASE
            WHEN sales_line_id IS NULL
              OR TRIM(sales_line_id) = ''
            THEN 1
            ELSE 0
        END
    ) AS missing_sales_line_id,

    SUM(
        CASE
            WHEN invoice_no IS NULL
              OR TRIM(invoice_no) = ''
            THEN 1
            ELSE 0
        END
    ) AS missing_invoice_no,

    SUM(
        CASE
            WHEN stock_code IS NULL
              OR TRIM(stock_code) = ''
            THEN 1
            ELSE 0
        END
    ) AS missing_stock_code,

    SUM(
        CASE
            WHEN customer_id IS NULL
              OR TRIM(customer_id) = ''
            THEN 1
            ELSE 0
        END
    ) AS missing_customer_id,

    SUM(
        CASE
            WHEN invoice_date IS NULL
            THEN 1
            ELSE 0
        END
    ) AS missing_invoice_date

FROM fact_sales;


-- ============================================================
-- 9. Referential integrity validation
-- ============================================================

SELECT
    SUM(
        CASE
            WHEN p.stock_code IS NULL
            THEN 1
            ELSE 0
        END
    ) AS fact_rows_without_product,

    SUM(
        CASE
            WHEN c.customer_id IS NULL
            THEN 1
            ELSE 0
        END
    ) AS fact_rows_without_customer,

    SUM(
        CASE
            WHEN d.date IS NULL
            THEN 1
            ELSE 0
        END
    ) AS fact_rows_without_date

FROM fact_sales f

LEFT JOIN dim_product p
    ON f.stock_code = p.stock_code

LEFT JOIN dim_customer c
    ON f.customer_id = c.customer_id

LEFT JOIN dim_date d
    ON f.invoice_date = d.date;


-- ============================================================
-- 10. Product dimension key uniqueness
-- Expected: zero returned rows
-- ============================================================

SELECT
    stock_code,
    COUNT(*) AS duplicate_count

FROM dim_product

GROUP BY
    stock_code

HAVING
    COUNT(*) > 1

ORDER BY
    duplicate_count DESC;


-- ============================================================
-- 11. Customer dimension key uniqueness
-- Expected: zero returned rows
-- ============================================================

SELECT
    customer_id,
    COUNT(*) AS duplicate_count

FROM dim_customer

GROUP BY
    customer_id

HAVING
    COUNT(*) > 1

ORDER BY
    duplicate_count DESC;


-- ============================================================
-- 12. Date dimension key uniqueness
-- Expected: zero returned rows
-- ============================================================

SELECT
    date,
    COUNT(*) AS duplicate_count

FROM dim_date

GROUP BY
    date

HAVING
    COUNT(*) > 1

ORDER BY
    duplicate_count DESC;


-- ============================================================
-- 13. Validate date dimension continuity
-- Expected:
-- total_calendar_days = expected_calendar_days
-- ============================================================

SELECT
    MIN(date) AS minimum_date,
    MAX(date) AS maximum_date,
    COUNT(*) AS total_calendar_days,

    DATEDIFF(
        MAX(date),
        MIN(date)
    ) + 1 AS expected_calendar_days,

    CASE
        WHEN COUNT(*) =
             DATEDIFF(MAX(date), MIN(date)) + 1
        THEN 'PASS'
        ELSE 'FAIL'
    END AS continuity_status

FROM dim_date;


-- ============================================================
-- 14. Validate transaction measure behavior
-- ============================================================

SELECT
    transaction_type,

    SUM(
        CASE
            WHEN transaction_type = 'SALE'
             AND sold_quantity <= 0
            THEN 1
            ELSE 0
        END
    ) AS invalid_sale_quantity_rows,

    SUM(
        CASE
            WHEN transaction_type = 'SALE'
             AND gross_sales_amount <= 0
            THEN 1
            ELSE 0
        END
    ) AS invalid_sale_amount_rows,

    SUM(
        CASE
            WHEN transaction_type = 'CANCELLATION'
             AND cancelled_quantity <= 0
            THEN 1
            ELSE 0
        END
    ) AS invalid_cancellation_quantity_rows,

    SUM(
        CASE
            WHEN transaction_type = 'CANCELLATION'
             AND cancellation_amount < 0
            THEN 1
            ELSE 0
        END
    ) AS invalid_cancellation_amount_rows

FROM fact_sales

GROUP BY
    transaction_type

ORDER BY
    transaction_type;


-- ============================================================
-- 15. Validate UNKNOWN customer member
-- ============================================================

SELECT
    COUNT(*) AS unknown_customer_dimension_rows

FROM dim_customer

WHERE customer_id = 'UNKNOWN';


-- ============================================================
-- 16. Validate non-product business codes
-- ============================================================

SELECT
    stock_code,
    canonical_product_description,
    is_non_product_code

FROM dim_product

WHERE stock_code IN (
    'POST',
    'DOT',
    'D',
    'M',
    'AMAZONFEE',
    'BANK CHARGES',
    'B'
)

ORDER BY
    stock_code;


-- ============================================================
-- 17. Validate Gold table counts
-- ============================================================

SELECT
    'fact_sales' AS table_name,
    COUNT(*) AS total_records
FROM fact_sales

UNION ALL

SELECT
    'dim_product',
    COUNT(*)
FROM dim_product

UNION ALL

SELECT
    'dim_customer',
    COUNT(*)
FROM dim_customer

UNION ALL

SELECT
    'dim_date',
    COUNT(*)
FROM dim_date

UNION ALL

SELECT
    'gold_data_quality_summary',
    COUNT(*)
FROM gold_data_quality_summary;


-- ============================================================
-- 18. Gold quality summary validation
-- ============================================================

SELECT
    total_silver_rows,
    unknown_customer_rows,
    unknown_product_rows,
    sale_rows,
    cancellation_rows,
    inventory_adjustment_rows,
    non_commercial_rows,
    bad_debt_net_loss_amount,
    gold_processed_ts

FROM gold_data_quality_summary;


-- ============================================================
-- 19. Validate expected quality summary row count
-- Expected: 1
-- ============================================================

SELECT
    COUNT(*) AS quality_summary_rows,

    CASE
        WHEN COUNT(*) = 1
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM gold_data_quality_summary;